# Day 4 — ILT 1: Schema Evolution Concepts
### GlobalMart Data Engineering · 9:30 AM – 10:30 AM

---

## Session Objectives

By the end of this session you will be able to:
- Explain what schema evolution is and why it happens in production
- Identify the 4 types of schema change and their risk level
- Configure Auto Loader to handle new columns automatically
- Use `mergeSchema` and `overwriteSchema` on Delta writes
- Describe the `_rescued_data` safety net column

---

## Agenda

| Time | Topic |
|------|-------|
| 9:30 | What is schema evolution and why it matters |
| 9:40 | 4 types of schema change — risk levels |
| 9:50 | How Auto Loader handles schema changes |
| 10:05 | How Delta Lake handles schema changes |
| 10:20 | GlobalMart scenario walkthrough |
| 10:25 | Best practices + Q&A |

---
## 1. What Is Schema Evolution?

**Schema** = the structure of your data: column names, data types, nullability.

**Schema Evolution** = the schema changes over time as the business changes.

### Why Does It Happen?

```
DAY 1:  orders.csv has 8 columns
        OrderID, CustomerID, OrderDate, ShippingDate, ...

DAY 90: Supplier adds a new column
        OrderID, CustomerID, OrderDate, ShippingDate, ..., PremiumFlag

DAY 180: Marketing renames a column
         OrderChannel → SalesChannel

DAY 270: Finance changes a data type
         TotalAmount (string) → TotalAmount (decimal)
```

> In the real world, schemas **always** evolve. Your pipeline must handle it gracefully — or it will crash.

### The Risk Without Schema Handling

| Scenario | Without Schema Handling | With Schema Handling |
|----------|------------------------|----------------------|
| New column added | Pipeline crashes | New column added automatically |
| Column removed | Pipeline crashes | Missing column shows as null |
| Type changes | Silent data corruption | Error caught early |
| Column renamed | Wrong data in wrong column | Handled via column mapping |

---
## 2. The 4 Types of Schema Change

### Type 1 — Additive (New Column Added) 🟢 Low Risk

```
BEFORE:  OrderID | CustomerID | OrderDate
AFTER:   OrderID | CustomerID | OrderDate | PremiumFlag  ← new
```

- Safest change — backward compatible
- Existing rows get `null` for the new column
- Auto Loader + Delta handle this automatically with the right config
- **Most common in production**

---

### Type 2 — Subtractive (Column Removed) 🟡 Medium Risk

```
BEFORE:  OrderID | CustomerID | OrderDate | LegacyCode
AFTER:   OrderID | CustomerID | OrderDate             ← LegacyCode gone
```

- Downstream dashboards/queries that reference `LegacyCode` will break
- Bronze layer should KEEP the column (fill with null) — never drop Bronze columns
- Silver/Gold can drop it intentionally

---

### Type 3 — Rename (Column Renamed) 🔴 High Risk

```
BEFORE:  OrderID | OrderChannel
AFTER:   OrderID | SalesChannel   ← renamed from OrderChannel
```

- Looks like: old column removed + new column added → double impact
- Without column mapping: all historical `OrderChannel` data appears to disappear
- With Delta Column Mapping (`mode=name`): rename is tracked in Delta metadata
- **Requires careful coordination between source and pipeline teams**

---

### Type 4 — Type Change (Data Type Changed) 🔴 High Risk

```
BEFORE:  TotalAmount STRING  → "1234.56"
AFTER:   TotalAmount DECIMAL → 1234.56
```

- Widening (int → long, string → string): usually safe
- Narrowing (long → int, decimal → string): data loss risk
- Auto Loader's `inferSchema` may infer differently run-to-run
- Best practice: **pin your schema explicitly** for critical columns

---
## 3. How Auto Loader Handles Schema Changes

Auto Loader has a dedicated option: `cloudFiles.schemaEvolutionMode`

### The 4 Modes

| Mode | Behaviour | Use When |
|------|-----------|----------|
| `addNewColumns` | Adds new columns automatically, updates schema location | **Default — use this in production** |
| `rescue` | Keeps existing schema strict, stores unexpected data in `_rescued_data` | Strict pipelines that need to audit unexpected fields |
| `failOnNewColumns` | Stream fails if a new column appears | Compliance pipelines where any change must be reviewed |
| `none` | Ignores new columns — they are silently dropped | Legacy pipelines where schema is fully controlled externally |

---

### Mode: `addNewColumns` (what we use)

```python
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)   # schema stored here
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # ← key option
    .option("header", "true")
    .option("inferSchema", "true")
    .load(raw_path)
)
```

**What happens when a new column appears:**
1. Auto Loader detects `PremiumFlag` in the new file
2. Updates the schema stored at `schema_path`
3. Stream restarts automatically with the new schema
4. New column is written to Bronze Delta table
5. Historical rows get `null` for `PremiumFlag`

---

### The Schema Location — Why It Matters

```
_autoloader/schemas/orders_demo/
    ├── schema   ← current inferred schema (JSON)
    └── ...
```

- Auto Loader stores the schema here between runs
- On restart, it loads from this location — not re-infers from scratch
- If you delete this folder (Clean Start), schema is re-inferred fresh
- Keeps schema consistent across restarts — prevents silent drift

---
## 4. The `_rescued_data` Column

When `schemaEvolutionMode = rescue`, Auto Loader adds a special column: `_rescued_data`

```python
# With rescue mode:
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.schemaEvolutionMode", "rescue")  # ← rescue mode
    ...
)
```

**What it does:**
```
Incoming row:  OrderID=001 | CustomerID=C01 | NewUnexpectedColumn=XYZ

Schema says:   OrderID, CustomerID  (does NOT know about NewUnexpectedColumn)

Result:
  OrderID           = 001
  CustomerID        = C01
  _rescued_data     = {"NewUnexpectedColumn": "XYZ"}   ← stored as JSON string
```

**Use case:** Audit pipelines where you want to capture ALL incoming data without breaking the schema, then review `_rescued_data` to decide whether to add the column officially.

> Think of `_rescued_data` as a **safety net** — nothing is lost, you just decide later what to do with it.

---
## 5. How Delta Lake Handles Schema Changes

Even after Auto Loader handles the incoming schema, the **Delta table** also needs to accept the change.

### Option 1 — `mergeSchema=true` (Additive Only)

```python
# On writeStream:
query = (
    raw_stream.writeStream
    .format("delta")
    .option("mergeSchema", "true")   # ← new columns are added to the Delta table
    .option("checkpointLocation", checkpoint_path)
    .start(output_path)
)
```

| Behaviour | Detail |
|-----------|--------|
| New column in incoming data | Added to Delta table schema |
| Existing columns | Unchanged |
| Historical rows | Get `null` for the new column |
| Column removed from source | Delta keeps the column (fills null) |

**This is what we use in the Auto Loader demo.**

---

### Option 2 — `overwriteSchema=true` (Destructive)

```python
# On write (batch):
df.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .save(output_path)
```

- **Replaces the entire table schema** — dangerous on production tables
- Historical data is also overwritten
- Only use during development or full reloads
- Cannot be used on writeStream — batch only

---

### Option 3 — Delta Column Mapping (Rename Support)

```sql
-- First, enable column mapping on the table:
ALTER TABLE delta.`/path/to/table`
SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
);

-- Then rename the column:
ALTER TABLE delta.`/path/to/table`
RENAME COLUMN OrderChannel TO SalesChannel;
```

- No data rewrite — just updates Delta metadata
- Historical data instantly accessible under the new name
- Works because Delta stores a mapping: `SalesChannel → physical_col_id_7`
- Requires `minReaderVersion=2`, `minWriterVersion=5`

---
## 6. GlobalMart Scenario — Schema Evolution in Practice

### The Scenario

```
Week 1:  Supplier sends orders.csv with 8 columns
         Auto Loader ingests → Bronze has 8 columns + _ingested_at + _source_file

Week 4:  Supplier adds PremiumFlag column to flag high-value orders
         orders_week4.csv now has 9 columns

Week 8:  Marketing renames OrderChannel → SalesChannel in the source system
         orders_week8.csv uses SalesChannel instead of OrderChannel
```

### What Happens With Our Pipeline

```
Week 4 — New Column (PremiumFlag):
  Auto Loader:  schemaEvolutionMode=addNewColumns → detects PremiumFlag
                Updates schema at _autoloader/schemas/
                Stream auto-restarts with new schema
  Delta:        mergeSchema=true → adds PremiumFlag column to Bronze
  Result:       Weeks 1-3 rows: PremiumFlag = null
                Week 4+ rows:   PremiumFlag = True/False  ✅

Week 8 — Column Rename (OrderChannel → SalesChannel):
  Auto Loader:  Sees OrderChannel GONE + SalesChannel NEW
                Adds SalesChannel as new column
                OrderChannel remains in schema (fills null for new rows)
  Bronze:       Both columns exist — historical data preserved
  Action needed: RENAME via Delta Column Mapping to merge them cleanly  ⚠
```

### The Key Insight

> **Auto Loader + mergeSchema handles additive changes automatically.**  
> **Renames require a manual ALTER TABLE RENAME COLUMN step.**

In [ ]:
# ─── ILLUSTRATIVE: Full Auto Loader setup with schema evolution ────────────────
# This is a reference snippet — not meant to run standalone.
# Shows how all the schema evolution options fit together.

"""
from pyspark.sql.functions import current_timestamp, input_file_name

# Step 1: Read stream with schema evolution enabled
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)        # persisted schema
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")    # additive evolution
    .option("header",      "true")
    .option("inferSchema", "true")
    .load(raw_path)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

# Step 2: Write to Delta with mergeSchema
query = (
    raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema",        "true")   # Bronze table accepts new columns
    .trigger(processingTime="30 seconds")
    .start(output_path)
)
"""

print("Schema evolution config — key options:")
print()
print("  cloudFiles.schemaEvolutionMode = addNewColumns")
print("     → Auto Loader adds new columns to schema automatically")
print()
print("  mergeSchema = true")
print("     → Delta table accepts new columns without failing the write")
print()
print("  Together: pipeline survives new columns with zero manual intervention.")

In [ ]:
# ─── ILLUSTRATIVE: Detect schema drift by inspecting the schema location ────────
# After Auto Loader runs, you can check what schema it inferred.
# Useful to audit unexpected column additions.

"""
import json

schema_file = f"{schema_path}/schema"
schema_json  = dbutils.fs.head(schema_file)
schema_dict  = json.loads(schema_json)

print("Current Auto Loader inferred schema:")
for field in schema_dict.get('fields', []):
    print(f"  {field['name']:<35} {field['type']}")
"""

print("To audit schema drift:")
print("  1. Read the schema file from the schemaLocation folder")
print("  2. Compare column count before vs after a new file lands")
print("  3. Alert if unexpected columns appear")

---
## 7. Summary — Decision Framework

```
Schema change detected in incoming data
          |
          ▼
    What type of change?
    |
    ├── New column added
    │       → addNewColumns + mergeSchema=true
    │       → Handled automatically ✅
    │
    ├── Column removed from source
    │       → Keep column in Bronze (fill null)
    │       → Drop intentionally in Silver ✅
    │
    ├── Column renamed
    │       → Short term: two columns exist (old=null for new rows, new=null for old rows)
    │       → Long term: ALTER TABLE RENAME COLUMN with columnMapping ✅
    │
    └── Data type changed
            → Cast explicitly in Silver transformation
            → Never rely on inferSchema for type changes ✅
```

---

## Key Takeaways

1. **Schemas always evolve** — build your pipeline to expect it, not avoid it
2. **addNewColumns** = the right Auto Loader mode for 95% of production cases
3. **mergeSchema=true** on every writeStream — no reason NOT to have it
4. **Never drop Bronze columns** — absorb the change, decide in Silver
5. **_rescued_data** is your audit safety net for strict pipelines
6. **Type changes** are the sneakiest — always cast explicitly in Silver

---

## Discussion Questions

1. *You receive a new CSV with 5 extra columns you've never seen. What happens with `addNewColumns`? What happens with `failOnNewColumns`?*

2. *A column is renamed from `OrderChannel` to `SalesChannel`. A junior engineer suggests just dropping the old column and adding the new one in Bronze. What's the problem with this approach?*

3. *Why is `overwriteSchema=true` dangerous on a production Bronze table?*

4. *In the GlobalMart pipeline, `shipping_tier.csv` has a column `Cost (Rs)`. What problem does this cause and how do we fix it?*